In [9]:
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
import pandas as pd
from datetime import datetime
import os
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

class InventoryManager:
    def __init__(self, root):
        self.root = root
        self.root.title('부품 재고 관리 시스템')
        self.root.geometry('1600x900')
        self.manual_mode = False
        self.inventory_file = '부품_재고현황.xlsx'
        self.history_file = 'inventory_history.xlsx'
        self.load_inventory()
        self.build_ui()

    def load_inventory(self):
        self.raw = pd.read_excel(self.inventory_file)
        self.inventory = self.raw.iloc[2:, [0,1,2,3,4,5,6]].copy()
        self.inventory.columns = ['순번','신품번','구품번','품명','재고수량','공정진행','입고수량']
        self.inventory['재고수량'] = pd.to_numeric(self.inventory['재고수량'], errors='coerce').fillna(0)

    def build_ui(self):
        self.selected_item = None
        frame = tk.Frame(self.root)
        frame.pack(fill='both', expand=True)

        tk.Button(frame, text='납품명세서 업로드', command=self.upload_delivery).pack(pady=5)
        tk.Button(frame, text='수동모드', command=self.toggle_manual).pack(pady=5)
        tk.Button(frame, text='대시보드', command=self.dashboard).pack(pady=5)

        cols = ('순번','신품번','구품번','품명','재고수량','공정진행','입고수량')
        self.tree = ttk.Treeview(frame, columns=cols, show='headings')
        for col in cols:
            self.tree.heading(col, text=col)
        self.tree.pack(fill='both', expand=True)
        self.tree.bind('<Double-1>', self.edit_inventory)
        tk.Button(frame, text='재고 수정', command=self.edit_inventory).pack(pady=5)
        self.refresh_tree()

    def refresh_tree(self):
        for i in self.tree.get_children():
            self.tree.delete(i)
        for _, row in self.inventory.iterrows():
            self.tree.insert('', 'end', values=list(row))

    def upload_delivery(self):
        file = filedialog.askopenfilename(filetypes=[('Excel files','*.xlsx *.xls')])
        if not file:
            return
        delivery = pd.read_excel(file)
        for _, row in delivery.iterrows():
            part = row['신품번']
            qty = row['납품수량']
            idx = self.inventory[self.inventory['신품번']==part].index
            if len(idx):
                self.inventory.loc[idx,'재고수량'] -= qty
        self.save_inventory()
        self.refresh_tree()
        messagebox.showinfo('완료','재고 업데이트 완료')

    def save_inventory(self):
        date = datetime.today().strftime('%Y-%m-%d')
        self.raw.columns.values[6] = date
        self.raw.iloc[2:,4] = self.inventory['재고수량'].values
        self.raw.to_excel(self.inventory_file,index=False)
        log = self.inventory[['품번','재고수량']].copy()
        log['날짜']=date
        if os.path.exists(self.history_file):
            old = pd.read_excel(self.history_file)
            log = pd.concat([old,log])
        log.to_excel(self.history_file,index=False)

    def toggle_manual(self):
        if not self.manual_mode:
            # 커스텀 비밀번호 창 생성
            pw_window = tk.Toplevel(self.root)
            pw_window.title('관리자 인증')
            pw_window.geometry('300x150') # 입력창 크기 지정
            
            tk.Label(pw_window, text='비밀번호를 입력하세요').pack(pady=10)
            pw_entry = tk.Entry(pw_window, show='*', width=20)
            pw_entry.pack(pady=5)
            pw_entry.focus_set() # 바로 타이핑 가능하게 설정

            def check_pw():
                if pw_entry.get() == 'admin1234':
                    self.manual_mode = True
                    messagebox.showwarning('수동모드', '관리자 모드 활성화')
                    pw_window.destroy()
                else:
                    messagebox.showerror('오류', '비밀번호 불일치')
            
            tk.Button(pw_window, text='확인', command=check_pw).pack(pady=10)
            
        else:
            self.manual_mode = False
            messagebox.showwarning('수동모드', '관리자 모드 종료')

    def edit_inventory(self, event=None):
        if not self.manual_mode:
            messagebox.showwarning('경고','수동모드를 먼저 켜세요')
            return
        selected = self.tree.selection()
        if not selected:
            messagebox.showwarning('경고','수정할 품목 선택')
            return
        item = self.tree.item(selected[0])
        values = item['values']
        edit = tk.Toplevel(self.root)
        edit.title('재고 수정')
        tk.Label(edit,text=f'품번: {values[0]}').pack()
        tk.Label(edit,text=f'현재 재고: {values[2]}').pack()
        qty_entry = tk.Entry(edit)
        qty_entry.pack()
        tk.Label(edit,text='사유').pack()
        reason_entry = tk.Entry(edit)
        reason_entry.pack()
        def save_edit():
            new_qty = int(qty_entry.get())
            reason = reason_entry.get()
            idx = self.inventory[self.inventory['신품번']==values[0]].index
            old_qty = self.inventory.loc[idx,'재고수량'].iloc[0]
            self.inventory.loc[idx,'재고수량'] = new_qty
            log = pd.DataFrame([{'날짜':datetime.today().strftime('%Y-%m-%d'),'품번':values[0],'변경전':old_qty,'변경후':new_qty,'사유':reason}])
            if os.path.exists('manual_log.xlsx'):
                old = pd.read_excel('manual_log.xlsx')
                log = pd.concat([old,log])
            log.to_excel('manual_log.xlsx',index=False)
            self.save_inventory()
            self.refresh_tree()
            edit.destroy()
            messagebox.showinfo('완료','수정 저장 완료')
        tk.Button(edit,text='저장',command=save_edit).pack()

    def dashboard(self):
        if not os.path.exists(self.history_file):
            messagebox.showerror('오류','이력 없음')
            return
        hist = pd.read_excel(self.history_file)
        summary = hist.groupby('날짜')['재고수량'].sum()
        summary.plot(kind='line')
        plt.title('재고 추이')
        plt.show()

root = tk.Tk()
app = InventoryManager(root)
root.mainloop()